# 🌸 Flowers Knowledge RAG Pipeline
A step-by-step Retrieval-Augmented Generation (RAG) demo using LangChain + Google Gemini + ChromaDB

## 1️⃣ Install Dependencies

In [ ]:
%pip install langchain langchain-community langchain-google-genai langchain-chroma langchain-text-splitters google-generativeai python-dotenv

## 2️⃣ Load Environment Variables

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
api_key = os.environ.get("GOOGLE_API_KEY")
print("API Key loaded:", "✅" if api_key else "❌ Not found")

## 3️⃣ Import Libraries

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

print("All libraries imported successfully ✅")

## 4️⃣ Load the Knowledge Base (flowers.txt)

In [ ]:
loader = TextLoader("flowers.txt", encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")
print(f"Total characters: {len(documents[0].page_content)}")

## 5️⃣ Split Documents into Chunks

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
docs = splitter.split_documents(documents)

print(f"Total chunks created: {len(docs)}")
print(f"\nSample chunk:\n{docs[0].page_content[:300]}...")

## 6️⃣ Create Vector Database with Gemini Embeddings

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vector database created successfully ✅")

## 7️⃣ Initialize Gemini LLM

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0.3,
    google_api_key=api_key
)

print("Gemini LLM initialized ✅")

## 8️⃣ Build the RAG Function

In [ ]:
def ask_rag(question):
    # Step 1: Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # Step 2: Build prompt
    prompt = f"""
You are a factual question-answering assistant.
Use ONLY the information provided in the context.
If the answer is not present, say: "I don't know".

Context:
{context}

Question: {question}

Answer:
"""
    # Step 3: Get answer from LLM
    response = llm.invoke(prompt)
    content = response.content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content

print("RAG function ready ✅")

## 9️⃣ Ask Questions!

In [ ]:
question = "What is the purpose of a flower?"
answer = ask_rag(question)
print(f"Q: {question}")
print(f"A: {answer}")

In [ ]:
question = "What are the parts of a flower?"
answer = ask_rag(question)
print(f"Q: {question}")
print(f"A: {answer}")

In [ ]:
question = "What is pollination?"
answer = ask_rag(question)
print(f"Q: {question}")
print(f"A: {answer}")